# Lesson 8.5: Logistic Regression: Predicting Yes or No

Predict pass or fail from study hours with logistic regression for the lesson's 20 students: coefficients, odds ratio, predicted probability and a confusion matrix.

Part of *Statistics from Scratch*. Run each cell with **Shift + Enter**, top to bottom.

## The question

A tutor has the study hours of 20 students and whether each passed a test (1 = passed, 0 = failed). Studying helps, but not for sure: one student passed after 1.75 hours and another failed after 4. What is the probability of passing after a given number of hours, and how much does each extra hour improve the odds?

If a package is missing on your computer, run the next cell once. In Google Colab everything is already installed.

In [ ]:
%pip install -q numpy scipy pandas statsmodels matplotlib

## Step 1: Enter the pass/fail data

Hours studied for 20 students and whether each passed (1) or failed (0). An outcome that is only yes or no needs logistic regression, not a straight line.

In [1]:
import numpy as np  # Load NumPy, used later for exponentials.
import pandas as pd  # Load pandas, which holds data in tables (DataFrames).
import statsmodels.formula.api as smf  # Load statsmodels' formula interface, which includes logistic regression.

hours = [0.5, 0.75, 1, 1.25, 1.5, 1.75, 2, 2.25, 2.5, 2.75,  # Hours studied by the first ten students (the list continues on the next line).
         3, 3.25, 3.5, 4, 4.25, 4.5, 4.75, 5, 5.5, 6]  # Hours studied by the last ten students. The two lines are one list.
passed = [0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1]  # Whether each student passed: 1 for pass, 0 for fail, in the same order as the hours.
df = pd.DataFrame({'hours': hours, 'passed': passed})  # Put both lists in one table with named columns so the formula can refer to them.

print('students passed:', df.passed.sum(), 'of', len(df))  # Print how many students passed out of 20. Ten of each makes the groups balanced.

students passed: 10 of 20


**What this means.** Ten of the 20 students passed, so the data are balanced. The outcome is yes/no, so a straight-line model would predict impossible probabilities (below 0 or above 1). That is why we use logistic regression.

## Step 2: Fit the logistic regression

logit models the log-odds of passing as a straight line in hours. The coefficients are on the log-odds scale: the slope is how much the log-odds rise for one extra hour.

In [2]:
model = smf.logit('passed ~ hours', data=df).fit(disp=0)  # logit fits a logistic regression: 'passed ~ hours' models the log-odds of passing as a straight line in hours. disp=0 hides the fitting log.

print(model.params.round(4))  # Print the intercept and slope on the log-odds scale, rounded to 4 decimals.
print()  # Print an empty line to separate the two outputs.
print(model.pvalues.round(4))  # Print the p-value of each coefficient. The slope's p-value tests whether hours help at all.
print(f'log-likelihood = {model.llf:.4f}')  # The log-likelihood measures how well the model fits the pass/fail pattern; it is mainly used to compare models.

Intercept   -4.2542
hours        1.4487
dtype: float64

Intercept    0.0213
hours        0.0164
dtype: float64
log-likelihood = -7.7461


**What this means.** The slope is 1.4487 on the log-odds scale: each extra hour raises the log-odds of passing by about 1.45. Its p-value is 0.0164, below 0.05, so hours really are linked to passing. The intercept (-4.2542) is the log-odds at 0 hours, a very low chance.

## Step 3: Odds ratios and a predicted probability

Exponentiating a coefficient turns it into an odds ratio: the factor by which the odds of passing are multiplied for each extra hour. Putting a number of hours through the model gives the probability of passing.

In [3]:
odds_ratio = np.exp(model.params['hours'])  # Exponentiate the slope. exp turns log-odds into an odds ratio: the factor the odds of passing are multiplied by per extra hour.
ci = np.exp(model.conf_int().loc['hours'])  # Do the same for the ends of the 95% confidence interval for the slope, to get an interval for the odds ratio.

print(f'odds ratio per hour = {odds_ratio:.2f}')  # Print the odds ratio.
print(f'95% CI: ({ci[0]:.2f}, {ci[1]:.2f})')  # Print its confidence interval. If it excludes 1, hours really change the odds.

p3 = model.predict(pd.DataFrame({'hours': [3]}))[0]  # Ask the model for the probability of passing at 3 hours. The new data must be a table with the same column name.
print(f'probability of passing after 3 hours = {p3:.3f}')  # Print that probability.

odds ratio per hour = 4.26
95% CI: (1.30, 13.90)
probability of passing after 3 hours = 0.523


**What this means.** The odds ratio is 4.26: each extra hour multiplies the odds of passing by about 4.3. The 95% interval (1.30, 13.90) is wide, with only 20 students, but it stays above 1. At 3 hours the model gives a 0.523 chance of passing, close to a coin flip.

## Step 4: Confusion matrix at a 0.5 cut-off

To turn probabilities into yes/no predictions, predict 'pass' when the probability is 0.5 or more. The matrix counts correct and wrong predictions in each group.

In [4]:
prob = model.predict(df)  # Get each student's predicted probability of passing from the model.
pred = (prob >= 0.5).astype(int)  # Turn probabilities into yes/no predictions: predict a pass when the probability is 0.5 or higher.

table = pd.crosstab(pred, df.passed, rownames=['predicted'], colnames=['actually passed'])  # Cross-tabulate predictions against what really happened. This confusion matrix counts right and wrong calls in each group.
print(table)  # Print the matrix.
print(f'accuracy = {(pred == df.passed).mean():.2f}')  # Accuracy is the fraction of students predicted correctly, the diagonal of the matrix divided by 20.

actually passed  0  1
predicted            
0                8  2
1                2  8
accuracy = 0.80


**What this means.** At a 0.5 cut-off, 8 students were correctly predicted to fail and 8 correctly predicted to pass; 2 passers were missed and 2 failers were wrongly called passers. Accuracy is 0.80. That is measured on the same students used to fit the model, so it is a little optimistic.

## So, what is the answer?

Yes, more study hours raise the chance of passing: each extra hour multiplies the odds of passing by about 4.3 (95% interval 1.30 to 13.90, p = 0.016), and a student who studies 3 hours has about a 52% chance. The model classified 80% of these 20 students correctly. Caution: with only 20 students the odds ratio is imprecise, and the 80% was checked on the same data used to fit the model.

## Now you try

A clinic tracks 18 patients with high blood pressure. For each it records average daily steps (in thousands) and whether blood pressure reached the target after three months (1 = yes, 0 = no). Do more steps raise the chance of reaching the target?

**Data:** steps = [2, 3, 3.5, 4, 4.5, 5, 5.5, 6, 6.5, 7, 7.5, 8, 8.5, 9, 9.5, 10, 11, 12]
ok = [0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1]

1. Fit a logistic regression of ok on steps and report the coefficients and the slope's p-value.
2. Compute the odds ratio per extra thousand steps.
3. Predict the probability of reaching the target at 8 thousand steps.
4. Compute the accuracy at a 0.5 cut-off.

*Hint:* Use smf.logit('ok ~ steps', data=df).fit(disp=0). Exponentiate the slope for the odds ratio, and compare predictions with 0.5 for accuracy.

In [ ]:
# Your turn: write your code here


### Solution (try first, then peek)

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

steps = [2, 3, 3.5, 4, 4.5, 5, 5.5, 6, 6.5, 7, 7.5, 8, 8.5, 9, 9.5, 10, 11, 12]
ok = [0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1]
df = pd.DataFrame({'steps': steps, 'ok': ok})
m = smf.logit('ok ~ steps', data=df).fit(disp=0)
print(m.params.round(3).to_dict(), 'p for steps =', m.pvalues['steps'].round(4))
print(f'odds ratio = {np.exp(m.params["steps"]):.2f}')
print(f'P(ok) at 8 = {m.predict(pd.DataFrame({"steps": [8]}))[0]:.3f}')
print(f'accuracy = {((m.predict(df) >= 0.5).astype(int) == df.ok).mean():.2f}')

{'Intercept': -3.953, 'steps': 0.584} p for steps = 0.0329
odds ratio = 1.79
P(ok) at 8 = 0.672
accuracy = 0.78


**Answer.** The slope is 0.584 on the log-odds scale (intercept -3.953) with p = 0.0329, so more steps are linked to reaching the target. The odds ratio is 1.79: each extra thousand steps multiplies the odds by about 1.8. At 8 thousand steps the model gives a 0.672 chance. Accuracy at a 0.5 cut-off is 0.78, measured on the same patients used to fit the model, so it is a bit optimistic.

## The same question in R, SPSS and Stata

**R**

```r
hours <- c(0.5, 0.75, 1, 1.25, 1.5, 1.75, 2, 2.25, 2.5, 2.75, 3, 3.25, 3.5, 4, 4.25, 4.5, 4.75, 5, 5.5, 6)  # Hours studied by the 20 students.
passed <- c(0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1)  # Whether each passed: 1 for pass, 0 for fail, in the same order.
fit <- glm(passed ~ hours, family = binomial)  # glm fits a generalised linear model. family = binomial makes it a logistic regression for a yes/no outcome.
summary(fit)  # Print the full results: coefficients on the log-odds scale, z values and p-values.
exp(coef(fit))  # exp(coef(fit)) turns the coefficients into odds ratios. The hours entry is the odds ratio per extra hour.
predict(fit, data.frame(hours = 3), type = "response")  # Predict the probability of passing at 3 hours. type = "response" gives a probability rather than log-odds.
table(predicted = fitted(fit) >= 0.5, actual = passed)  # Build the confusion matrix: predicted pass when the fitted probability is at least 0.5, against what actually happened.
```

How to read it: In summary, the hours row has Estimate 1.4487, z value 2.400 and Pr(>|z|) 0.0164, so hours are linked to passing. exp(coef) gives 4.26 for hours: the odds multiply by about 4.3 per hour. predict gives 0.523. The table shows 8 and 8 correct calls, 2 and 2 wrong.

**SPSS** (Analyze > Regression > Binary Logistic (Dependent: passed, Covariate: hours; Options: CI for Exp(B)))

```
DATA LIST FREE / hours passed.
BEGIN DATA
0.5 0  0.75 0  1 0  1.25 0  1.5 0  1.75 1  2 0  2.25 0  2.5 0  2.75 1
3 0  3.25 1  3.5 1  4 0  4.25 1  4.5 1  4.75 1  5 1  5.5 1  6 1
END DATA.
LOGISTIC REGRESSION VARIABLES passed
  /METHOD=ENTER hours
  /PRINT=CI(95)
  /CRITERIA=CUT(0.5).
```

How to read it: In Variables in the Equation read hours: B = 1.449, Sig. = .016 and Exp(B) = 4.26 with a 95% CI of about 1.30 to 13.90. The Constant is -4.254. The Classification Table at cut value 0.500 shows 8 and 8 correct, 2 and 2 wrong, Overall Percentage 80.0.

**Stata**

```
clear
set obs 20
generate hours = real(word("0.5 0.75 1 1.25 1.5 1.75 2 2.25 2.5 2.75 3 3.25 3.5 4 4.25 4.5 4.75 5 5.5 6", _n))
generate passed = real(word("0 0 0 0 0 1 0 0 0 1 0 1 1 0 1 1 1 1 1 1", _n))
logit passed hours
logistic passed hours
estat classification
margins, at(hours = 3)
```

How to read it: logit: hours Coef. 1.449, P>|z| 0.016 and _cons -4.254. logistic: Odds Ratio 4.26 with a 95% interval of about 1.30 to 13.90. estat classification: Correctly classified 80.00%. margins: about .523, the chance of passing after 3 hours.